# Run PGS-adjusted Cox + pan-cancer testosterone

Two analyses launched from the same notebook so the cluster paths stay in one place:

1. **`cox_pgs_adjusted.py`** — nested-model PGS-adjusted Cox per `(endpoint, landmark, target_lab, PGS)`: baseline-lab, joint (lab + PGS), and PGS-alone arms.
2. **`pan_cancer_testosterone_univariate.py`** — sanity check for the time-to-platinum work: univariate Cox per testosterone aggregation on pan-cancer males, with cancer-type adjustment.

**Before running:** activate the `survlatent_ode` conda env. Edit the `Paths` cell for local copies of the data.

## Paths

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path("/data/gusev/USERS/jpconnor/code/CAIA")
SURVIVAL_DIR = PROJECT_ROOT / "COMPASS" / "survival_analysis"

# === cox_pgs_adjusted.py inputs ===
AGGREGATED_PATTERN = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis/prediction_inputs/aggregated_landmark{landmark}.csv"
GERMLINE_PATH      = Path("/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/clinical_and_genomic_features/complete_germline_data_df.csv.gz")
PGS_OUTPUT_DIR     = Path("/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/survival_analysis")

# === pan_cancer_testosterone_univariate.py inputs ===
LABS_CSV            = Path("/data/gusev/USERS/jpconnor/data/PROFILE/OncDRS/ALL_2025_03/OUTPT_LAB_RESULTS_LABS.csv")
SURVIVAL_CSV        = Path("/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/time-to-event_analysis/death_met_surv_df.csv.gz")
CANCER_TYPE_CSV     = Path("/data/gusev/USERS/jpconnor/data/clinical_text_embedding_project/clinical_and_genomic_features/cancer_type_df.csv.gz")
FIRST_TREATMENT_CSV = Path("/data/gusev/USERS/jpconnor/data/PROFILE/robust_VTE_pred_project_2025_03_cohort/data/follow_up_vte_df_cohort.csv")
PAN_OUTPUT_DIR      = SURVIVAL_DIR / "results" / "pan_cancer_testosterone"

LANDMARKS = [0, 90]
ENDPOINTS = ["platinum", "death"]

os.chdir(PROJECT_ROOT)
PGS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAN_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# BLAS thread caps so lifelines/numpy don't oversubscribe cores in a shared session.
os.environ["OMP_NUM_THREADS"]      = "1"
os.environ["MKL_NUM_THREADS"]      = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"]  = "1"

print("cwd:             ", os.getcwd())
print("pgs_output_dir:  ", PGS_OUTPUT_DIR)
print("pan_output_dir:  ", PAN_OUTPUT_DIR)

## 1. PGS-adjusted Cox (baseline-lab + joint + PGS-alone)

Writes one CSV per landmark to `<pgs_output_dir>/cox/landmark_{N}/both/cox_pgs_adjusted.csv`. Each row covers a `(endpoint, target_lab, PGS)` combination; the PGS-alone fit is constant per `(endpoint, PGS)` and broadcast across the two target_lab rows.

In [ ]:
import subprocess, time

cmd = [
    "python", str(SURVIVAL_DIR / "cox_pgs_adjusted.py"),
    "--aggregated-csv-pattern", AGGREGATED_PATTERN,
    "--germline-path",          str(GERMLINE_PATH),
    "--output-dir",             str(PGS_OUTPUT_DIR),
    "--landmarks",              *[str(L) for L in LANDMARKS],
    "--endpoints",              *ENDPOINTS,
]
print(" ".join(cmd))
t0 = time.time()
rc = subprocess.call(cmd)
print(f"\nrc={rc}  elapsed={(time.time() - t0)/60:.1f} min")

### Inspect: top PGS per `(endpoint, lab_name)`

Sorts on `p_value_pgs` (joint-model PGS effect) and shows the PGS-alone columns alongside, so you can read "does PGS add on top of lab" (`p_value_pgs`) and "PGS-alone vs lab-alone" (compare `hazard_ratio_pgs_alone_per_sd` vs `hazard_ratio_baseline_per_sd`) in one row.

In [ ]:
import pandas as pd

pgs_frames = []
for L in LANDMARKS:
    p = PGS_OUTPUT_DIR / "cox" / f"landmark_{L}" / "both" / "cox_pgs_adjusted.csv"
    if not p.exists():
        print(f"[missing] {p}")
        continue
    df = pd.read_csv(p)
    df.insert(0, "landmark_days", L)
    pgs_frames.append(df)

if pgs_frames:
    pgs_all = pd.concat(pgs_frames, ignore_index=True)
    cols = [
        "landmark_days", "endpoint", "lab_name", "pgs",
        "n_patients_used", "n_events_used",
        "hazard_ratio_baseline_per_sd", "p_value_baseline",
        "hazard_ratio_pgs_alone_per_sd", "p_value_pgs_alone", "q_value_pgs_alone",
        "hazard_ratio_per_sd", "p_value",
        "hazard_ratio_pgs_per_sd", "p_value_pgs", "q_value_pgs",
        "coef_feature_delta",
    ]
    cols = [c for c in cols if c in pgs_all.columns]
    top = (
        pgs_all.dropna(subset=["p_value_pgs"])
               .sort_values(["landmark_days", "endpoint", "lab_name", "p_value_pgs"])
               .groupby(["landmark_days", "endpoint", "lab_name"], as_index=False)
               .head(5)
    )
    display(top[cols])
else:
    print("No PGS results found yet \u2014 run section 1 above first.")

## 2. Pan-cancer testosterone × OS univariate Cox

Cohort: pan-cancer males. For each landmark in `LANDMARKS`, fits Cox per testosterone aggregation (`__mean/__min/__max/__last/__delta/__slope`) adjusted for age, `Testosterone__n_observations`, and cancer-type dummies (rare types merged into `OTHER`, threshold = 30).

Writes `<pan_output_dir>/pan_cancer_testosterone_univariate_landmark{N}.csv`.

**Before trusting effect sizes:** check the printed TEST_TYPE_DESCR distribution and unit consistency. The script does not canonicalize testosterone units.

In [ ]:
cmd = [
    "python", str(SURVIVAL_DIR / "pan_cancer_testosterone_univariate.py"),
    "--labs-csv",            str(LABS_CSV),
    "--survival-csv",        str(SURVIVAL_CSV),
    "--cancer-type-csv",     str(CANCER_TYPE_CSV),
    "--first-treatment-csv", str(FIRST_TREATMENT_CSV),
    "--output-dir",          str(PAN_OUTPUT_DIR),
    "--landmarks",           *[str(L) for L in LANDMARKS],
]
print(" ".join(cmd))
t0 = time.time()
rc = subprocess.call(cmd)
print(f"\nrc={rc}  elapsed={(time.time() - t0)/60:.1f} min")

### Inspect: testosterone features ranked by p-value within each landmark

In [ ]:
pan_frames = []
for L in LANDMARKS:
    p = PAN_OUTPUT_DIR / f"pan_cancer_testosterone_univariate_landmark{L}.csv"
    if not p.exists():
        print(f"[missing] {p}")
        continue
    pan_frames.append(pd.read_csv(p))

if pan_frames:
    pan_all = pd.concat(pan_frames, ignore_index=True)
    cols = [
        "landmark_days", "feature", "feature_stat",
        "n_patients_used", "n_patients_observed", "n_events_used",
        "hazard_ratio_per_sd", "ci_lower", "ci_upper",
        "p_value", "q_value", "note",
    ]
    cols = [c for c in cols if c in pan_all.columns]
    display(pan_all.sort_values(["landmark_days", "p_value"], na_position="last")[cols])
else:
    print("No pan-cancer results found yet \u2014 run section 2 above first.")